# NB05B — MULTI-SEED SHUFFLE ROBUSTNESS | NIR-HUEVOS 2026

**Extension of NB05_WAVELENGTH_ORDER_ABLATION, requested to address a peer-review comment.**

## Purpose
NB05 tested the `shuffled` wavelength-order condition using a single fixed target-independent
permutation (seed `52026`). A reviewer noted this limits how confidently the ablation result
generalizes ("recurrent networks are unsuitable for spectral data" should not rest on one shuffle).

NB05B repeats the **shuffled condition only** (the `original` and `reversed` conditions are
unaffected by this critique and are NOT re-run) across **several additional random permutation
seeds**, using the exact frozen NB04 recipe (architecture, preprocessing, epochs) and the exact
frozen NB02 egg-disjoint splits — nothing about the confirmatory pipeline changes.

## Critical controls (identical to NB05)
- Same 5 frozen egg-disjoint outer folds from NB02.
- Same architecture capacity, optimizer, learning rate, batch size, preprocessing and final
  epoch count already selected in NB04 (no retuning).
- Same final training seeds: **2026, 2027, 2028**.
- Preprocessing/scaling always computed in the true physical wavelength order first; the
  permutation is applied only after train-only preprocessing and scaling.
- Outer-test eggs are touched only for evaluation.
- No retuning after permuting wavelength order.

## What's new here vs. NB05
- `NEW_SHUFFLE_SEEDS` below lists the additional permutation seeds (default: 4, distinct from
  `52026` and from each other). Each is an independent draw from
  `np.random.default_rng(seed).permutation(331)`.
- Runtime scales linearly with the number of seeds in `NEW_SHUFFLE_SEEDS`. On the L4 GPU used
  for NB05, the shuffled condition alone (4 models x 5 folds x 3 training seeds = 60 fits) took
  roughly **40-45 minutes per permutation seed**, dominated by SimpleRNN. Four additional seeds
  is therefore roughly **2.5-3 hours** total. **Trim `NEW_SHUFFLE_SEEDS` to 2 if you need a
  faster turnaround** — even 2 additional seeds (3 total with the original) is enough to report
  a mean +/- range instead of a single point estimate, which is the core of the reviewer's ask.
- The notebook is checkpointed/resumable exactly like NB05: if your Colab session disconnects,
  rerunning resumes from the last completed (seed, fold, model, training-seed) combination.

## Compute
GPU required. L4 preferred (same as NB05).


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# Imports, paths, fixed protocol, and GPU gate
from pathlib import Path
from datetime import datetime, timezone
import gc, hashlib, json, os, platform, random, shutil, subprocess, sys, time, warnings

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_DETERMINISTIC_OPS'] = '1'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('/content/drive/MyDrive/NIR_HUEVOS_PAPER_REBUILD_2026')
RAW_DIR = PROJECT_ROOT / '01_DATA_RAW'
SPLIT_DIR = PROJECT_ROOT / '03_SPLITS_FROZEN'
NB04_DIR = PROJECT_ROOT / '05_RESULTS' / 'NB04_DEEP_LEARNING_BENCHMARK'
NB05_DIR = PROJECT_ROOT / '05_RESULTS' / 'NB05_WAVELENGTH_ORDER_ABLATION'
RESULT_DIR = PROJECT_ROOT / '05_RESULTS' / 'NB05B_MULTISEED_SHUFFLE_ABLATION'
FIG_DIR = PROJECT_ROOT / '06_FIGURES' / 'NB05B_MULTISEED_SHUFFLE_ABLATION'
ZIP_DIR = PROJECT_ROOT / '05_RESULTS' / 'ZIP_PACKAGES'
NOTEBOOKS_DIR = PROJECT_ROOT / '04_NOTEBOOKS'
CHECKPOINT_DIR = RESULT_DIR / '_CHECKPOINT'

for p in [RESULT_DIR, FIG_DIR, ZIP_DIR, NOTEBOOKS_DIR, CHECKPOINT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATA_FILE = RAW_DIR / 'dataset_egg_storage_RAW.csv'
OUTER_FILE = SPLIT_DIR / 'outer_group_assignment_seed2026.csv'
SPLIT_MANIFEST_FILE = SPLIT_DIR / 'split_manifest.json'
NB04_PROTOCOL_FILE = NB04_DIR / 'NB04_protocol.json'
NB04_SUMMARY_FILE = NB04_DIR / 'NB04_run_summary.json'
NB04_SELECTED_FILE = NB04_DIR / 'NB04_selected_configurations.csv'
NB04_PRED_FILE = NB04_DIR / 'NB04_oof_predictions_seedwise.csv'
NB05_POOLED_SEEDMEAN_FILE = NB05_DIR / 'NB05_pooled_seedmean_metrics_all_orders.csv'

EXPECTED_DATASET_SHA256 = 'cd5021c555ae6b57f892549c574599cef75edf87f58b3f7f4d246ade9327d15e'
EXPECTED_SPLIT_MANIFEST_SHA256 = 'fbeb8fa19d522cd91bee875bf5731cda264475da27bc7e93c25ca0d6f0f33717'
EXPECTED_NB04_REVISION = 'NB04_v1_fixed_capacity_inner_preprocessing_epoch_selection'

NOTEBOOK_FILENAME = 'NB05B_MULTISEED_SHUFFLE_ABLATION.ipynb'
RUN_REVISION = 'NB05B_v1_multiseed_shuffle_robustness'
PACKAGE_SCHEMA = 'NIR-HUEVOS standardized result package v1.3'

MODELS = ['ANN', 'SimpleRNN', 'LSTM', 'BiLSTM']
FINAL_SEEDS = [2026, 2027, 2028]
ORIGINAL_SHUFFLE_SEED = 52026  # the seed already reported in the manuscript (NB05)

# ---- EDIT THIS LIST IF YOU NEED A FASTER RUN (2 seeds is an acceptable minimum) ----
NEW_SHUFFLE_SEEDS = [11017, 24601, 73819, 90210]
# -------------------------------------------------------------------------------

RECURRENT_UNITS = 64
ANN_HIDDEN = [64, 32]
DENSE_AFTER_RECURRENT = 32
DROPOUT = 0.20
LEARNING_RATE = 1e-3
BATCH_SIZE = 32
LOSS = 'mse'
SG_WINDOW = 11
SG_POLYORDER = 2

REQUIRE_GPU = True
RESUME_IF_AVAILABLE = True

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__)
print('GPU devices:', gpus)
if REQUIRE_GPU:
    assert len(gpus) > 0, (
        'NB05B requires a GPU. In Colab choose Runtime > Change runtime type > GPU, then rerun.'
    )
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass


TensorFlow: 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# Integrity gate: dataset, frozen splits, and approved NB04 evidence (identical checks to NB05)

def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

required = [
    DATA_FILE, OUTER_FILE, SPLIT_MANIFEST_FILE,
    NB04_PROTOCOL_FILE, NB04_SUMMARY_FILE, NB04_SELECTED_FILE, NB04_PRED_FILE
]
for p in required:
    assert p.exists(), f'Missing required input: {p}'

dataset_sha = sha256_file(DATA_FILE)
split_manifest_sha = sha256_file(SPLIT_MANIFEST_FILE)
assert dataset_sha == EXPECTED_DATASET_SHA256, 'Dataset hash changed.'
assert split_manifest_sha == EXPECTED_SPLIT_MANIFEST_SHA256, 'Frozen split manifest hash changed.'

split_manifest = json.loads(SPLIT_MANIFEST_FILE.read_text(encoding='utf-8'))
for file_name, expected_hash in split_manifest['files'].items():
    p = SPLIT_DIR / file_name
    assert p.exists(), f'Missing frozen split file: {p}'
    assert sha256_file(p) == expected_hash, f'Frozen split changed: {file_name}'

nb04_protocol = json.loads(NB04_PROTOCOL_FILE.read_text(encoding='utf-8'))
nb04_summary = json.loads(NB04_SUMMARY_FILE.read_text(encoding='utf-8'))
assert nb04_protocol['run_revision'] == EXPECTED_NB04_REVISION
assert nb04_summary['status'] == 'COMPLETED'
assert nb04_protocol['dataset_sha256'] == dataset_sha
assert nb04_protocol['frozen_split_manifest_sha256'] == split_manifest_sha

selected_nb04 = pd.read_csv(NB04_SELECTED_FILE)
original_nb04 = pd.read_csv(NB04_PRED_FILE)
assert len(selected_nb04) == 20
assert len(original_nb04) == 7920

# Reference: original-seed (52026) shuffled-condition pooled metrics from NB05, for comparison
assert NB05_POOLED_SEEDMEAN_FILE.exists(), (
    f'Expected NB05 output not found: {NB05_POOLED_SEEDMEAN_FILE}. '
    'Run/keep NB05 results in place before running NB05B.'
)
nb05_pooled = pd.read_csv(NB05_POOLED_SEEDMEAN_FILE)
assert set(nb05_pooled['order_condition']) >= {'original', 'reversed', 'shuffled'}

# Checkpoint identity
STATE_FILE = CHECKPOINT_DIR / 'NB05B_checkpoint_state.json'
resume_valid = False
if RESUME_IF_AVAILABLE and STATE_FILE.exists():
    try:
        state = json.loads(STATE_FILE.read_text(encoding='utf-8'))
        resume_valid = (
            state.get('run_revision') == RUN_REVISION and
            state.get('dataset_sha256') == dataset_sha and
            state.get('split_manifest_sha256') == split_manifest_sha and
            state.get('nb04_revision') == EXPECTED_NB04_REVISION and
            state.get('new_shuffle_seeds') == NEW_SHUFFLE_SEEDS
        )
    except Exception:
        resume_valid = False

if not resume_valid:
    for p in RESULT_DIR.iterdir():
        if p.name not in ['_CHECKPOINT']:
            if p.is_dir():
                shutil.rmtree(p)
            else:
                p.unlink()
    if CHECKPOINT_DIR.exists():
        for p in CHECKPOINT_DIR.iterdir():
            if p.is_dir():
                shutil.rmtree(p)
            else:
                p.unlink()
    state = {
        'run_revision': RUN_REVISION,
        'dataset_sha256': dataset_sha,
        'split_manifest_sha256': split_manifest_sha,
        'nb04_revision': EXPECTED_NB04_REVISION,
        'new_shuffle_seeds': NEW_SHUFFLE_SEEDS,
        'started_at_utc': datetime.now(timezone.utc).isoformat(),
        'status': 'IN_PROGRESS'
    }
    STATE_FILE.write_text(json.dumps(state, indent=2), encoding='utf-8')
    print('Fresh NB05B run initialized.')
else:
    print('Valid NB05B checkpoint detected — completed fits will be skipped.')

print('PASS — dataset, frozen splits, and approved NB04 evidence verified.')


Fresh NB05B run initialized.
PASS — dataset, frozen splits, and approved NB04 evidence verified.


In [4]:
# Load data in physical wavelength order; build one permutation per NEW shuffle seed

df = pd.read_csv(DATA_FILE)
outer = pd.read_csv(OUTER_FILE)

spectral_cols = [c for c in df.columns if c.startswith('Spectra_')]
def wavelength_from_col(c):
    return float(c.replace('Spectra_', ''))
spectral_cols = sorted(spectral_cols, key=wavelength_from_col)

wavelengths = np.array([wavelength_from_col(c) for c in spectral_cols], dtype=np.float64)
X_all = df[spectral_cols].to_numpy(dtype=np.float32)
y_all = df['storage_days'].to_numpy(dtype=np.float32)

assert X_all.shape == (660, 331)
assert df['sample'].nunique() == 30
assert df['storage_days'].nunique() == 22
assert wavelengths[0] == 740 and wavelengths[-1] == 1070
assert np.all(np.diff(wavelengths) == 1)

N_FEATURES = len(spectral_cols)
ORIGINAL_IDX = np.arange(N_FEATURES, dtype=int)

# Original NB05 permutation, reconstructed only to guarantee our new seeds are distinct from it
ORIGINAL_SHUFFLE_IDX = np.random.default_rng(ORIGINAL_SHUFFLE_SEED).permutation(N_FEATURES)

SHUFFLE_IDX_BY_SEED = {}
for s in NEW_SHUFFLE_SEEDS:
    idx = np.random.default_rng(s).permutation(N_FEATURES)
    assert np.array_equal(np.sort(idx), ORIGINAL_IDX)
    assert not np.array_equal(idx, ORIGINAL_IDX)
    assert not np.array_equal(idx, ORIGINAL_SHUFFLE_IDX), f'Seed {s} collided with the original NB05 permutation — pick a different seed.'
    SHUFFLE_IDX_BY_SEED[s] = idx

# Pairwise distinctness across the new seeds themselves
for i, si in enumerate(NEW_SHUFFLE_SEEDS):
    for sj in NEW_SHUFFLE_SEEDS[i+1:]:
        assert not np.array_equal(SHUFFLE_IDX_BY_SEED[si], SHUFFLE_IDX_BY_SEED[sj]), f'Seeds {si} and {sj} produced the same permutation.'

order_rows = []
for s, idx in SHUFFLE_IDX_BY_SEED.items():
    for new_position, source_index in enumerate(idx):
        order_rows.append({
            'shuffle_seed': s,
            'new_position_0based': int(new_position),
            'source_index_0based': int(source_index),
            'wavelength_nm_at_new_position': float(wavelengths[source_index])
        })
order_map = pd.DataFrame(order_rows)
order_map.to_csv(RESULT_DIR / 'NB05B_wavelength_order_map.csv', index=False)

print(f'Verified {len(NEW_SHUFFLE_SEEDS)} new shuffle permutations, all distinct from each other and from seed {ORIGINAL_SHUFFLE_SEED}.')
for s in NEW_SHUFFLE_SEEDS:
    print(f'  seed {s}: first 8 wavelengths ->', wavelengths[SHUFFLE_IDX_BY_SEED[s][:8]].astype(int).tolist())


Verified 4 new shuffle permutations, all distinct from each other and from seed 52026.
  seed 11017: first 8 wavelengths -> [853, 933, 862, 1026, 851, 849, 824, 957]
  seed 24601: first 8 wavelengths -> [816, 1049, 1047, 856, 961, 962, 886, 952]
  seed 73819: first 8 wavelengths -> [829, 788, 1067, 777, 1036, 764, 1040, 1052]
  seed 90210: first 8 wavelengths -> [978, 783, 817, 915, 1006, 758, 798, 982]


## Ablation rule (identical to NB05)

For every model x outer fold x shuffle seed:
- preprocessing and epoch count are imported from the approved NB04 selection;
- the preprocessor is fitted using **all 24 outer-training eggs only**;
- preprocessing and StandardScaler operate in the original physical wavelength order;
- only then is the transformed feature matrix permuted using that shuffle seed's index;
- the model is trained from scratch for the frozen number of epochs;
- seeds 2026, 2027 and 2028 are repeated for training stochasticity;
- evaluation occurs once on the six unseen outer-test eggs.

No inner cross-validation is repeated, and no retuning follows permutation — exactly as in NB05.


In [5]:
# Preprocessing, metrics, model builders, and reproducibility utilities (identical to NB05)

class NeuralSpectralPreprocessor:
    def __init__(self, name, sg_window=11, sg_polyorder=2):
        self.name = name
        self.sg_window = sg_window
        self.sg_polyorder = sg_polyorder
        self.reference_ = None
        self.scaler_ = None

    def _base_transform(self, X):
        X = np.asarray(X, dtype=np.float64)
        if self.name == 'raw':
            return X.copy()
        if self.name == 'snv':
            mu = X.mean(axis=1, keepdims=True)
            sd = X.std(axis=1, ddof=1, keepdims=True)
            sd = np.where(sd < 1e-12, 1.0, sd)
            return (X - mu) / sd
        if self.name == 'msc':
            if self.reference_ is None:
                raise RuntimeError('MSC reference must be fitted on training data first.')
            ref = self.reference_
            ref_mean = ref.mean()
            ref_centered = ref - ref_mean
            denom = np.dot(ref_centered, ref_centered)
            out = np.empty_like(X, dtype=np.float64)
            for i, x in enumerate(X):
                x_mean = x.mean()
                b = np.dot(ref_centered, x - x_mean) / denom
                if abs(b) < 1e-12:
                    b = 1.0
                a = x_mean - b * ref_mean
                out[i] = (x - a) / b
            return out
        if self.name == 'sg_smooth':
            return savgol_filter(
                X, window_length=self.sg_window, polyorder=self.sg_polyorder,
                deriv=0, axis=1, mode='interp'
            )
        if self.name == 'sg_deriv1':
            return savgol_filter(
                X, window_length=self.sg_window, polyorder=self.sg_polyorder,
                deriv=1, delta=1.0, axis=1, mode='interp'
            )
        raise ValueError(f'Unknown preprocessing: {self.name}')

    def fit(self, X):
        X = np.asarray(X, dtype=np.float64)
        if self.name == 'msc':
            self.reference_ = X.mean(axis=0)
        base = self._base_transform(X)
        self.scaler_ = StandardScaler().fit(base)
        return self

    def transform(self, X):
        if self.scaler_ is None:
            raise RuntimeError('Preprocessor must be fitted first.')
        base = self._base_transform(X)
        return self.scaler_.transform(base).astype(np.float32)

    def fit_transform(self, X):
        return self.fit(X).transform(X)


def metrics_dict(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    err = y_pred - y_true
    ae = np.abs(err)
    return {
        'MAE_days': float(mean_absolute_error(y_true, y_pred)),
        'RMSE_days': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'R2': float(r2_score(y_true, y_pred)),
        'bias_days': float(np.mean(err)),
        'median_AE_days': float(np.median(ae)),
        'within_1d_pct': float(100*np.mean(ae <= 1.0)),
        'within_2d_pct': float(100*np.mean(ae <= 2.0)),
        'within_3d_pct': float(100*np.mean(ae <= 3.0)),
    }


def set_all_seeds(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


def build_model(model_name, n_features=331):
    if model_name == 'ANN':
        inp = keras.Input(shape=(n_features,), name='spectrum')
        x = layers.Dense(ANN_HIDDEN[0], activation='relu')(inp)
        x = layers.Dropout(DROPOUT)(x)
        x = layers.Dense(ANN_HIDDEN[1], activation='relu')(x)
        x = layers.Dropout(DROPOUT)(x)
        out = layers.Dense(1, activation='linear')(x)
    else:
        inp = keras.Input(shape=(n_features, 1), name='wavelength_sequence')
        if model_name == 'SimpleRNN':
            x = layers.SimpleRNN(RECURRENT_UNITS, return_sequences=False)(inp)
        elif model_name == 'LSTM':
            x = layers.LSTM(RECURRENT_UNITS, return_sequences=False)(inp)
        elif model_name == 'BiLSTM':
            x = layers.Bidirectional(layers.LSTM(RECURRENT_UNITS, return_sequences=False))(inp)
        else:
            raise ValueError(model_name)
        x = layers.Dropout(DROPOUT)(x)
        x = layers.Dense(DENSE_AFTER_RECURRENT, activation='relu')(x)
        x = layers.Dropout(DROPOUT)(x)
        out = layers.Dense(1, activation='linear')(x)

    model = keras.Model(inp, out, name=model_name)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=LOSS,
        metrics=[keras.metrics.MeanAbsoluteError(name='mae')]
    )
    return model


def shape_for_model(X, model_name):
    X = np.asarray(X, dtype=np.float32)
    return X if model_name == 'ANN' else X[..., np.newaxis]


def apply_shuffle(X, shuffle_seed):
    return X[:, SHUFFLE_IDX_BY_SEED[shuffle_seed]]


def get_outer_train_test_eggs(outer_fold):
    test_eggs = set(outer.loc[outer['outer_fold'] == outer_fold, 'sample'].tolist())
    train_eggs = set(df['sample'].unique()) - test_eggs
    assert len(train_eggs) == 24 and len(test_eggs) == 6
    assert not (train_eggs & test_eggs)
    return train_eggs, test_eggs


In [6]:
# Resumable checkpoints for the multi-seed shuffled fits
CP_METRICS = CHECKPOINT_DIR / 'NB05B_multiseed_outer_fold_seed_metrics.csv'
CP_PRED = CHECKPOINT_DIR / 'NB05B_multiseed_oof_predictions_seedwise.csv'

def load_cp(path):
    return pd.read_csv(path) if path.exists() and path.stat().st_size > 0 else pd.DataFrame()

def save_cp(df_, path):
    tmp = path.with_suffix(path.suffix + '.tmp')
    df_.to_csv(tmp, index=False)
    os.replace(tmp, path)

multiseed_metrics = load_cp(CP_METRICS)
multiseed_pred = load_cp(CP_PRED)

print('Checkpoint rows:', {'metrics': len(multiseed_metrics), 'predictions': len(multiseed_pred)})


Checkpoint rows: {'metrics': 0, 'predictions': 0}


In [ ]:
# Main NB05B loop: shuffled condition only, across NEW_SHUFFLE_SEEDS
run_start = time.perf_counter()
expected_pairs = set(zip(df['sample'].astype(int), df['storage_days'].astype(int)))

for shuffle_seed in NEW_SHUFFLE_SEEDS:
    print(f'\n{"="*18} SHUFFLE SEED {shuffle_seed} {"="*18}')
    for outer_fold in range(1, 6):
        print(f'\n----- outer fold {outer_fold}/5 -----')
        train_eggs, test_eggs = get_outer_train_test_eggs(outer_fold)

        train_mask = df['sample'].isin(train_eggs).to_numpy()
        test_mask = df['sample'].isin(test_eggs).to_numpy()

        X_train_raw = X_all[train_mask]
        y_train = y_all[train_mask]
        X_test_raw = X_all[test_mask]
        y_test = y_all[test_mask]
        test_meta = df.loc[test_mask, ['sample', 'storage_days']].reset_index(drop=True)

        assert len(X_train_raw) == 528 and len(X_test_raw) == 132

        for model_name in MODELS:
            cfg = selected_nb04[
                (selected_nb04['outer_fold'] == outer_fold) &
                (selected_nb04['model'] == model_name)
            ]
            assert len(cfg) == 1
            prep_name = str(cfg.iloc[0]['selected_preprocessing'])
            selected_epoch = int(cfg.iloc[0]['selected_epoch'])

            pp = NeuralSpectralPreprocessor(prep_name, SG_WINDOW, SG_POLYORDER)
            X_train_pp = pp.fit_transform(X_train_raw)
            X_test_pp = pp.transform(X_test_raw)

            Xtr_shuf = apply_shuffle(X_train_pp, shuffle_seed)
            Xte_shuf = apply_shuffle(X_test_pp, shuffle_seed)

            for seed in FINAL_SEEDS:
                already = False
                if not multiseed_metrics.empty:
                    already = (
                        (multiseed_metrics['shuffle_seed'].astype(int) == shuffle_seed) &
                        (multiseed_metrics['outer_fold'].astype(int) == outer_fold) &
                        (multiseed_metrics['model'] == model_name) &
                        (multiseed_metrics['seed'].astype(int) == seed)
                    ).any()

                if already:
                    continue

                tf.keras.backend.clear_session()
                gc.collect()
                set_all_seeds(seed)

                model = build_model(model_name, N_FEATURES)
                Xtr_m = shape_for_model(Xtr_shuf, model_name)
                Xte_m = shape_for_model(Xte_shuf, model_name)

                t0 = time.perf_counter()
                hist = model.fit(
                    Xtr_m, y_train,
                    epochs=selected_epoch,
                    batch_size=BATCH_SIZE,
                    verbose=0,
                    shuffle=True
                )
                train_time_s = time.perf_counter() - t0

                pred = model.predict(Xte_m, batch_size=BATCH_SIZE, verbose=0).reshape(-1)
                assert len(pred) == 132 and np.isfinite(pred).all()

                row = {
                    'shuffle_seed': shuffle_seed,
                    'outer_fold': outer_fold,
                    'model': model_name,
                    'seed': seed,
                    'selected_preprocessing': prep_name,
                    'selected_epoch': selected_epoch,
                    **metrics_dict(y_test, pred),
                    'train_time_s': float(train_time_s),
                    'final_train_loss': float(hist.history['loss'][-1]),
                    'final_train_mae': float(hist.history['mae'][-1])
                }
                multiseed_metrics = pd.concat([multiseed_metrics, pd.DataFrame([row])], ignore_index=True)

                pred_rows = test_meta.copy()
                pred_rows['shuffle_seed'] = shuffle_seed
                pred_rows['outer_fold'] = outer_fold
                pred_rows['model'] = model_name
                pred_rows['seed'] = seed
                pred_rows['y_pred'] = pred
                pred_rows['selected_preprocessing'] = prep_name
                pred_rows['selected_epoch'] = selected_epoch
                multiseed_pred = pd.concat([multiseed_pred, pred_rows], ignore_index=True)

                save_cp(multiseed_metrics, CP_METRICS)
                save_cp(multiseed_pred, CP_PRED)

                print(
                    f'  seed {shuffle_seed} | fold {outer_fold} | {model_name} | train_seed {seed}: '
                    f"MAE={row['MAE_days']:.4f} | RMSE={row['RMSE_days']:.4f} | R2={row['R2']:.4f} | train={train_time_s:.1f}s"
                )

                del model, hist, Xtr_m, Xte_m
                tf.keras.backend.clear_session()
                gc.collect()

run_elapsed_s = time.perf_counter() - run_start
print(f'\nNB05B loop wall time this session: {run_elapsed_s/60:.1f} min')



================== SHUFFLE SEED 11017 ==================

----- outer fold 1/5 -----
  seed 11017 | fold 1 | ANN | train_seed 2026: MAE=2.5502 | RMSE=3.1879 | R2=0.7475 | train=7.2s
  seed 11017 | fold 1 | ANN | train_seed 2027: MAE=2.4579 | RMSE=3.1316 | R2=0.7563 | train=5.5s


  seed 11017 | fold 1 | ANN | train_seed 2028: MAE=2.4031 | RMSE=3.0931 | R2=0.7623 | train=5.4s
  seed 11017 | fold 1 | SimpleRNN | train_seed 2026: MAE=4.8089 | RMSE=5.6790 | R2=0.1987 | train=141.9s
  seed 11017 | fold 1 | SimpleRNN | train_seed 2027: MAE=5.0316 | RMSE=5.9354 | R2=0.1248 | train=143.4s
  seed 11017 | fold 1 | SimpleRNN | train_seed 2028: MAE=5.1542 | RMSE=5.9899 | R2=0.1086 | train=143.3s
  seed 11017 | fold 1 | LSTM | train_seed 2026: MAE=3.8930 | RMSE=4.7587 | R2=0.4374 | train=23.5s
  seed 11017 | fold 1 | LSTM | train_seed 2027: MAE=5.1213 | RMSE=6.1207 | R2=0.0692 | train=23.3s
  seed 11017 | fold 1 | LSTM | train_seed 2028: MAE=3.8224 | RMSE=4.7296 | R2=0.4442 | train=23.3s
  seed 11017 | fold 1 | BiLSTM | train_seed 2026: MAE=4.7406 | RMSE=5.5355 | R2=0.2387 | train=13.3s
  seed 11017 | fold 1 | BiLSTM | train_seed 2027: MAE=4.8589 | RMSE=5.5740 | R2=0.2281 | train=13.3s
  seed 11017 | fold 1 | BiLSTM | train_seed 2028: MAE=4.6619 | RMSE=5.4142 | R2=0.2717 | 

In [ ]:
# Completeness gate
n_expected = len(NEW_SHUFFLE_SEEDS) * 5 * len(MODELS) * len(FINAL_SEEDS)
assert len(multiseed_metrics) == n_expected, f'Expected {n_expected} rows, got {len(multiseed_metrics)}'
assert not multiseed_metrics.duplicated(['shuffle_seed', 'outer_fold', 'model', 'seed']).any()
assert len(multiseed_pred) == n_expected * 132
assert not multiseed_pred.duplicated(['shuffle_seed', 'outer_fold', 'model', 'seed', 'sample', 'storage_days']).any()
assert np.isfinite(multiseed_pred['y_pred']).all()

for shuffle_seed in NEW_SHUFFLE_SEEDS:
    for model_name in MODELS:
        for seed in FINAL_SEEDS:
            g = multiseed_pred[
                (multiseed_pred['shuffle_seed'] == shuffle_seed) &
                (multiseed_pred['model'] == model_name) &
                (multiseed_pred['seed'] == seed)
            ]
            assert len(g) == 660
            assert set(zip(g['sample'].astype(int), g['storage_days'].astype(int))) == expected_pairs

multiseed_pred.to_csv(RESULT_DIR / 'NB05B_multiseed_shuffle_oof_predictions_seedwise.csv', index=False)
multiseed_metrics.to_csv(RESULT_DIR / 'NB05B_multiseed_shuffle_outer_fold_seed_metrics.csv', index=False)

print(f'PASS — {len(multiseed_pred)} OOF predictions across {len(NEW_SHUFFLE_SEEDS)} new shuffle seeds are complete.')


In [ ]:
# Pool to seed-mean, then to a single pooled MAE per (shuffle_seed, model); combine with the
# original NB05 shuffled-condition result (seed 52026) for a multiseed distribution

seedmean_pred = (
    multiseed_pred
    .groupby(['shuffle_seed', 'model', 'sample', 'storage_days', 'outer_fold'], as_index=False)
    .agg(y_pred=('y_pred', 'mean'))
)

pooled_rows = []
for (shuffle_seed, model_name), g in seedmean_pred.groupby(['shuffle_seed', 'model']):
    pooled_rows.append({
        'shuffle_seed': int(shuffle_seed),
        'model': model_name,
        **metrics_dict(g['storage_days'], g['y_pred'])
    })
pooled_new = pd.DataFrame(pooled_rows)

# Original NB05 shuffled-condition pooled seed-mean metrics (seed 52026), tagged for comparison
pooled_original_shuffled = nb05_pooled[nb05_pooled['order_condition'] == 'shuffled'].copy()
pooled_original_shuffled['shuffle_seed'] = ORIGINAL_SHUFFLE_SEED
pooled_original_shuffled = pooled_original_shuffled[pooled_new.columns.tolist()]

pooled_original_order = nb05_pooled[nb05_pooled['order_condition'] == 'original'][['model', 'MAE_days']].rename(
    columns={'MAE_days': 'MAE_original_order'}
)

pooled_all_shuffle = pd.concat([pooled_original_shuffled, pooled_new], ignore_index=True)
pooled_all_shuffle.to_csv(RESULT_DIR / 'NB05B_pooled_metrics_by_shuffle_seed.csv', index=False)

summary_rows = []
for model_name, g in pooled_all_shuffle.groupby('model'):
    orig_mae = float(pooled_original_order.loc[pooled_original_order['model'] == model_name, 'MAE_original_order'].iloc[0])
    summary_rows.append({
        'model': model_name,
        'n_shuffle_seeds': len(g),
        'shuffle_seeds': sorted(g['shuffle_seed'].astype(int).tolist()),
        'mean_MAE_days': float(g['MAE_days'].mean()),
        'sd_MAE_days': float(g['MAE_days'].std(ddof=1)),
        'min_MAE_days': float(g['MAE_days'].min()),
        'max_MAE_days': float(g['MAE_days'].max()),
        'MAE_original_wavelength_order': orig_mae,
        'n_shuffle_seeds_with_lower_MAE_than_original_order': int((g['MAE_days'] < orig_mae).sum())
    })
shuffle_multiseed_summary = pd.DataFrame(summary_rows)
shuffle_multiseed_summary.to_csv(RESULT_DIR / 'NB05B_shuffle_multiseed_summary.csv', index=False)

print('Multi-seed shuffled-condition summary (all shuffle seeds pooled per model):')
display(shuffle_multiseed_summary)


In [ ]:
# Diagnostic figure: MAE across shuffle permutation seeds per model, vs. original wavelength order
plt.figure(figsize=(8, 5))
model_order = MODELS
positions = np.arange(len(model_order))
for i, model_name in enumerate(model_order):
    g = pooled_all_shuffle[pooled_all_shuffle['model'] == model_name]
    jitter = (np.random.default_rng(0).random(len(g)) - 0.5) * 0.15
    plt.scatter(np.full(len(g), i) + jitter, g['MAE_days'], label=None, zorder=3)
    orig_mae = float(pooled_original_order.loc[pooled_original_order['model'] == model_name, 'MAE_original_order'].iloc[0])
    plt.scatter([i], [orig_mae], marker='D', color='black', zorder=4)
plt.xticks(positions, model_order)
plt.ylabel('Pooled OOF MAE (days)')
plt.title('NB05B: MAE across shuffle permutation seeds (circles) vs. original order (black diamond)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'NB05B_MAE_by_shuffle_seed.png', dpi=220)
plt.close()

print('Diagnostic figure saved to:', FIG_DIR)


## Interpretation gate

Do **not** treat `n_shuffle_seeds_with_lower_MAE_than_original_order` as a formal hypothesis test
— with only a handful of permutation seeds it is a descriptive count, not a p-value. What this
notebook adds is a **range** (mean +/- SD, min-max) for the shuffled-condition MAE per model,
which is what turns the single-seed NB05 result into a small-sample robustness check. If SimpleRNN
and LSTM MAE stay consistently lower under shuffling than under the original wavelength order
across all new seeds, that supports the NB05 conclusion; if it varies substantially seed-to-seed
(e.g., some new seeds land close to the original-order MAE), that itself is worth reporting as a
tempering of the original single-seed claim.

Formal paired inference across shuffle seeds (e.g., treating the shuffle seed as an additional
random factor) is left for a follow-up if the manuscript needs it — NB05B intentionally stays
descriptive, matching the scope NB05 itself used for order effects (formal inference was deferred
to NB06).


In [ ]:
# Freeze protocol and execution status
protocol = {
    'notebook': 'NB05B_MULTISEED_SHUFFLE_ABLATION',
    'notebook_filename': NOTEBOOK_FILENAME,
    'run_revision': RUN_REVISION,
    'dataset_sha256': dataset_sha,
    'frozen_split_manifest_sha256': split_manifest_sha,
    'source_NB04_revision': EXPECTED_NB04_REVISION,
    'original_NB05_shuffle_seed': ORIGINAL_SHUFFLE_SEED,
    'new_shuffle_seeds': NEW_SHUFFLE_SEEDS,
    'models': MODELS,
    'final_training_seeds': FINAL_SEEDS,
    'condition_repeated': 'shuffled only (original and reversed are unaffected by this check)',
    'preprocessing_rule': (
        'Use exact preprocessing selected by NB04 for each model x outer fold; '
        'fit on 24 outer-training eggs only; transform in physical wavelength order; '
        'apply the shuffle permutation only after preprocessing and StandardScaler.'
    ),
    'epoch_rule': 'Use exact NB04 selected epoch for each model x outer fold; no early stopping or retuning in NB05B.',
    'outer_test_used_for_selection': False,
    'retuning_after_shuffle': False,
}
(RESULT_DIR / 'NB05B_protocol.json').write_text(json.dumps(protocol, indent=2), encoding='utf-8')

summary = {
    'status': 'COMPLETED',
    'run_revision': RUN_REVISION,
    'n_models': len(MODELS),
    'n_new_shuffle_seeds': len(NEW_SHUFFLE_SEEDS),
    'n_outer_folds': 5,
    'n_training_seeds': len(FINAL_SEEDS),
    'new_final_fits_expected': len(NEW_SHUFFLE_SEEDS) * 5 * len(MODELS) * len(FINAL_SEEDS),
    'combined_seedwise_oof_predictions': int(len(multiseed_pred)),
    'n_eggs': int(df['sample'].nunique()),
    'n_days': int(df['storage_days'].nunique()),
    'no_inner_cv_repeated': True,
    'NB04_recipe_frozen': True,
    'zero_outer_group_leakage': True,
    'gpu_required': REQUIRE_GPU
}
(RESULT_DIR / 'NB05B_run_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')

state.update({
    'status': 'COMPLETED',
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'combined_seedwise_oof_predictions': int(len(multiseed_pred))
})
STATE_FILE.write_text(json.dumps(state, indent=2), encoding='utf-8')

print('NB05B status: COMPLETED')


## RESULT PACKAGE — AUTOMATIC ZIP

The final cell creates `NB05B_RESULTS_MULTISEED_SHUFFLE_ABLATION.zip` and downloads it to your
computer. The raw dataset is excluded. Send that ZIP back (same way as the NIR_HUEVOS_REVIEWER_ROUND
package) and the manuscript's Section 2.7 / 3.6 / Limitations point 8 will be updated with the
real multi-seed numbers in place of the current single-seed caveat.


In [ ]:
# STANDARDIZED RESULT PACKAGE v1.3 — NB05B_MULTISEED_SHUFFLE_ABLATION
from google.colab import files as colab_files

NB_CODE = 'NB05B_MULTISEED_SHUFFLE_ABLATION'
ZIP_NAME = 'NB05B_RESULTS_MULTISEED_SHUFFLE_ABLATION.zip'
PACKAGE_DIR = PROJECT_ROOT / '05_RESULTS' / '_PACKAGE_TMP' / NB_CODE

if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

dst_results = PACKAGE_DIR / '05_RESULTS_NB05B'
dst_results.mkdir(parents=True, exist_ok=True)
for p in sorted(RESULT_DIR.iterdir()):
    if p.name == '_CHECKPOINT':
        continue
    if p.is_file():
        shutil.copy2(p, dst_results / p.name)
    elif p.is_dir():
        shutil.copytree(p, dst_results / p.name)

dst_figs = PACKAGE_DIR / '06_FIGURES_NB05B'
shutil.copytree(FIG_DIR, dst_figs)

dst_splits = PACKAGE_DIR / '03_SPLITS_FROZEN'
dst_splits.mkdir(parents=True, exist_ok=True)
for p in sorted(SPLIT_DIR.glob('*')):
    if p.is_file():
        shutil.copy2(p, dst_splits / p.name)

dst_nb04 = PACKAGE_DIR / 'SOURCE_NB04_FROZEN'
dst_nb04.mkdir(parents=True, exist_ok=True)
for p in [NB04_PROTOCOL_FILE, NB04_SUMMARY_FILE, NB04_SELECTED_FILE]:
    shutil.copy2(p, dst_nb04 / p.name)

dst_nb05 = PACKAGE_DIR / 'SOURCE_NB05_FROZEN'
dst_nb05.mkdir(parents=True, exist_ok=True)
shutil.copy2(NB05_POOLED_SEEDMEAN_FILE, dst_nb05 / NB05_POOLED_SEEDMEAN_FILE.name)

notebook_file = NOTEBOOKS_DIR / NOTEBOOK_FILENAME
assert notebook_file.exists(), (
    f'Expected notebook source not found in Drive: {notebook_file}. '
    'Save NB05B_MULTISEED_SHUFFLE_ABLATION.ipynb inside 04_NOTEBOOKS before packaging.'
)
notebook_text = notebook_file.read_text(encoding='utf-8', errors='ignore')
assert RUN_REVISION in notebook_text, (
    'Drive notebook source does not contain the expected NB05B revision marker.'
)
notebook_sha = sha256_file(notebook_file)
shutil.copy2(notebook_file, PACKAGE_DIR / NOTEBOOK_FILENAME)

with open(PACKAGE_DIR / 'environment_packages.txt', 'w', encoding='utf-8') as f:
    f.write(f'Python: {sys.version}\n')
    f.write(f'Platform: {platform.platform()}\n')
    f.write(f'TensorFlow: {tf.__version__}\n\n')
    try:
        f.write(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True, stderr=subprocess.STDOUT))
    except Exception as e:
        f.write(f'pip freeze failed: {e}\n')

try:
    nvidia_info = subprocess.check_output(['nvidia-smi'], text=True, stderr=subprocess.STDOUT)
except Exception as e:
    nvidia_info = f'nvidia-smi unavailable: {e}'
(PACKAGE_DIR / 'hardware_info.txt').write_text(nvidia_info, encoding='utf-8')

run_manifest = {
    'project': 'NIR_HUEVOS_PAPER_REBUILD_2026',
    'notebook': NB_CODE,
    'notebook_filename': NOTEBOOK_FILENAME,
    'executed_notebook_source_sha256': notebook_sha,
    'run_revision': RUN_REVISION,
    'package_schema': PACKAGE_SCHEMA,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'dataset_file': DATA_FILE.name,
    'dataset_sha256': dataset_sha,
    'raw_dataset_included_in_zip': False,
    'frozen_split_manifest_sha256': split_manifest_sha,
    'source_NB04_revision': EXPECTED_NB04_REVISION,
    'original_NB05_shuffle_seed': ORIGINAL_SHUFFLE_SEED,
    'new_shuffle_seeds': NEW_SHUFFLE_SEEDS,
    'models': MODELS,
    'final_seeds': FINAL_SEEDS,
    'primary_metric': 'MAE_days',
    'python_version': sys.version,
    'tensorflow_version': tf.__version__,
    'platform': platform.platform(),
    'zip_name': ZIP_NAME
}
(PACKAGE_DIR / 'RUN_MANIFEST.json').write_text(json.dumps(run_manifest, indent=2), encoding='utf-8')

readme = f'''NB05B MULTI-SEED SHUFFLE ROBUSTNESS — NIR-HUEVOS 2026

STATUS: COMPLETED
RUN REVISION: {RUN_REVISION}

Design:
- repeats only the `shuffled` wavelength-order condition from NB05
- across {len(NEW_SHUFFLE_SEEDS)} additional permutation seeds: {NEW_SHUFFLE_SEEDS}
- original NB05 shuffled-condition result (seed {ORIGINAL_SHUFFLE_SEED}) is included for comparison
- NB04 preprocessing and epoch frozen per model x outer fold
- preprocessing/scaling fitted on outer-training eggs only
- shuffle applied after preprocessing
- same training seeds 2026/2027/2028
- no retuning
- see NB05B_shuffle_multiseed_summary.csv for the headline mean/SD/range per model

Raw dataset intentionally excluded.
'''
(PACKAGE_DIR / 'README.txt').write_text(readme, encoding='utf-8')

inventory = []
for p in sorted(PACKAGE_DIR.rglob('*')):
    if p.is_file() and p.name != 'PACKAGE_INVENTORY.json':
        inventory.append({
            'relative_path': str(p.relative_to(PACKAGE_DIR)),
            'size_bytes': p.stat().st_size,
            'sha256': sha256_file(p)
        })
(PACKAGE_DIR / 'PACKAGE_INVENTORY.json').write_text(json.dumps(inventory, indent=2), encoding='utf-8')

ZIP_DIR.mkdir(parents=True, exist_ok=True)
zip_path = ZIP_DIR / ZIP_NAME
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', root_dir=PACKAGE_DIR)

assert zip_path.exists() and zip_path.stat().st_size > 0
print('ZIP created:', zip_path)
print('ZIP size MB:', round(zip_path.stat().st_size / 1024**2, 2))

colab_files.download(str(zip_path))
